In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("netop/TeleQnA")

c:\Users\sadny\Documents\GitHub\fine_tuning_telco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\sadny\Documents\GitHub\fine_tuning_telco\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sadny\.cache\huggingface\hub\datasets--netop--TeleQnA. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admini

In [2]:
ds

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer', 'subject', 'explaination'],
        num_rows: 10000
    })
})

In [3]:
print(ds)
print(ds["test"].features)
print(ds["test"][0])

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer', 'subject', 'explaination'],
        num_rows: 10000
    })
})
{'question': Value('string'), 'choices': List(Value('string')), 'answer': Value('int64'), 'subject': Value('string'), 'explaination': Value('string')}
{'question': 'What is the purpose of the Nmfaf_3daDataManagement_Deconfigure service operation? [3GPP Release 18]', 'choices': ['To configure the MFAF to map data or analytics received by the MFAF to out-bound notification endpoints', 'To configure the MFAF to stop mapping data or analytics received by the MFAF to out-bound notification endpoints', 'To supply data or analytics from the MFAF to notification endpoints', 'To fetch data or analytics from the MFAF based on fetch instructions'], 'answer': 1, 'subject': 'Standards specifications', 'explaination': 'The Nmfaf_3daDataManagement_Deconfigure service operation is used to stop mapping data or analytics received by the MFAF to one or more o

In [4]:
for col in ds["test"].column_names:
    missing = sum(
        1 for x in ds["test"][col]
        if x is None or str(x).strip() == ""
    )
    print(f"{col}: {missing}")

question: 0
choices: 0
answer: 0
subject: 0
explaination: 0


In [5]:
from collections import Counter

subject_counts = Counter(ds["test"]["subject"])

print(f"Total subjects: {len(subject_counts)}\n")

for subject, count in subject_counts.most_common():
    print(f"{subject:<30} {count}")

Total subjects: 5

Research publications          4500
Standards specifications       2000
Research overview              2000
Standards overview             1000
Lexicon                        500


In [6]:
from collections import Counter

answer_counts = Counter(ds["test"]["answer"])

for ans, count in sorted(answer_counts.items()):
    print(ans, count)

0 2210
1 2179
2 2169
3 2153
4 1289


In [7]:
question_lengths = [
    len(q.split())
    for q in ds["test"]["question"]
]

print("Min:", min(question_lengths))
print("Max:", max(question_lengths))
print("Avg:", sum(question_lengths)/len(question_lengths))

Min: 2
Max: 38
Avg: 13.0327


In [8]:
import numpy as np

print(np.percentile(question_lengths,
                    [25, 50, 75, 90, 95, 99]))

[10. 13. 16. 19. 21. 26.]


In [9]:
from collections import Counter

choice_counts = Counter(
    len(c) for c in ds["test"]["choices"]
)

print(choice_counts)

Counter({5: 6441, 4: 3456, 3: 87, 2: 16})


In [10]:
lengths = [
    (i, len(q.split()))
    for i, q in enumerate(ds["test"]["question"])
]

for idx, length in sorted(lengths,
                          key=lambda x: x[1],
                          reverse=True)[:10]:
    print("\n")
    print("Length:", length)
    print(ds["test"][idx]["question"])



Length: 38
In an FDD (Frequency Division Duplex) protocol, what is the pilot/feedback overhead equivalent to, in terms of the number of additional UL (Uplink) pilot signals? (M is the number of transmit antennas, K is the number of UEs)


Length: 38
What is the capacity formula for a narrowband time-invariant MIMO channel? (nt is the number of transmit antennas, nr is the number of receive antennas, P is the transmit power, and N0 is the noise power spectral density)


Length: 37
What is the term used to describe the computation offloading problem in a multi-cell MEC (Mobile Edge Computing) network, where a dense deployment of radio access points facilitates high-bandwidth access to computational resources but also increases interference?


Length: 37
Which MAC (medium access control) protocol enables multiple users to transmit their signals via RISs (reconfigurable intelligent surfaces) relying on random contention-based multiple access protocols, where some control information exch

In [16]:
from collections import Counter

questions = ds["test"]["question"]

duplicates = [
    (q, c)
    for q, c in Counter(questions).items()
    if c > 1
]

print("Duplicate questions:", len(duplicates))

Duplicate questions: 25


In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-8B"
)

token_lengths = []

for q, exp in zip(
    ds["test"]["question"],
    ds["test"]["explaination"]
):
    text = q + "\n" + str(exp)

    token_lengths.append(
        len(tokenizer(text)["input_ids"])
    )

print("Average:", np.mean(token_lengths))
print("Max:", np.max(token_lengths))
print(
    np.percentile(
        token_lengths,
        [50, 90, 95, 99]
    )
)

Average: 45.0639
Max: 149
[43. 63. 71. 87.]


In [18]:
from collections import defaultdict

subject_exp = defaultdict(list)

for row in ds["test"]:
    subject_exp[row["subject"]].append(
        len(str(row["explaination"]).split())
    )

for subject, lengths in sorted(
    subject_exp.items(),
    key=lambda x: sum(x[1])/len(x[1]),
    reverse=True
):
    avg = sum(lengths)/len(lengths)
    print(f"{subject:<30} {avg:.1f}")

Standards specifications       20.9
Lexicon                        20.6
Research publications          20.6
Research overview              20.2
Standards overview             19.4


In [ ]:
print("Total samples:", len(ds["test"]))
print("Unique subjects:", len(set(ds["test"]["subject"])))
print("Duplicate questions:", len(duplicates))
print("Average question length:", np.mean(question_lengths))
print("Average explanation length:", np.mean(exp_lengths))
print("Maximum token length:", np.max(token_lengths))

Total samples: 10000
Unique subjects: 5
Duplicate questions: 25
Average question length: 13.0327
Average explanation length: 45.0639
Maximum token length: 149
